# Laboratorio 1 · Parte 1: Cuentas y acceso

En esta parte dejas listas las cuatro herramientas del curso y compruebas que funcionan:

| Herramienta | Qué se verifica |
|---|---|
| GitHub | Tu repositorio existe, se creó desde la plantilla y tiene tu nombre en el README. |
| Hugging Face | Tu token funciona: `whoami()` devuelve tu usuario. |
| Databricks | Tu token funciona y el volumen `raw` existe. |
| Colab | Este notebook queda guardado en tu repositorio, con sus salidas. |

Las celdas marcadas con `# TODO` son las que completas tú. Las demás ya están listas: solo ejecútalas.

> **Regla de oro:** los tokens nunca se escriben en el código. Se guardan como **secretos de Colab** (ícono 🔑 en la barra izquierda) y se leen con `userdata.get(...)`. Cuando Colab pregunte, activa el acceso del notebook a cada secreto.

## Preparación

In [ ]:
import requests
from google.colab import userdata

resultados = {}  # aquí se va guardando qué verificaciones pasaron

## 1. GitHub

1. Entra a la plantilla del curso: https://github.com/toms99/cenfotec-deep-learning---big-data
2. Clic en **Use this template → Create a new repository**. Déjalo **público**.
3. En tu repositorio, edita el `README.md` y escribe tu nombre completo donde dice *Estudiante*.
4. Abre este notebook en Colab desde tu repositorio: en Colab, **Archivo → Abrir cuaderno → GitHub**, y busca `lab1/parte1_verificacion.ipynb`.

Completa la celda siguiente con tus datos y ejecuta la verificación.

In [ ]:
# TODO: escribe tu repositorio (usuario/nombre-del-repositorio) y tu nombre
GITHUB_REPO = ""
NOMBRE = ""

In [ ]:
base = f"https://raw.githubusercontent.com/{GITHUB_REPO}/HEAD"

r = requests.get(f"{base}/README.md", timeout=30)
assert r.status_code == 200, "No encontré tu README. ¿El repositorio es público y escribiste bien usuario/nombre?"
assert NOMBRE.strip() and NOMBRE.lower() in r.text.lower(), "Tu README todavía no tiene tu nombre."

r = requests.get(f"{base}/lab1/parte1_verificacion.ipynb", timeout=30)
assert r.status_code == 200, "Tu repositorio no tiene lab1/: ¿lo creaste con «Use this template»?"

resultados["GitHub"] = True
print(f"✅ GitHub: {GITHUB_REPO}")

## 2. Hugging Face

1. Crea tu cuenta en https://huggingface.co/join
2. Ve a **Settings → Access Tokens → Create new token**, tipo **Read**.
3. En Colab, abre 🔑 **Secretos**, agrega `HF_TOKEN` con el token como valor y activa el acceso del notebook.

La función `whoami()` de la biblioteca `huggingface_hub` le pregunta a Hugging Face a quién pertenece un token. Si responde con tu usuario, el token funciona.

In [ ]:
from huggingface_hub import whoami

hf_token = userdata.get("HF_TOKEN")

# TODO: llama a whoami() con tu token y guarda la respuesta en usuario_hf
usuario_hf = None

In [ ]:
assert usuario_hf and "name" in usuario_hf, "whoami() no devolvió un usuario. Revisa la celda anterior."

resultados["Hugging Face"] = True
print(f"✅ Hugging Face: {usuario_hf['name']}")

## 3. Databricks Free Edition

1. Crea tu cuenta en https://www.databricks.com/learn/free-edition
2. Abre el **SQL Editor** y ejecuta:
   ```sql
   CREATE SCHEMA IF NOT EXISTS workspace.dlbd;
   CREATE VOLUME IF NOT EXISTS workspace.dlbd.raw;
   ```
3. Crea un token: tu perfil (arriba a la derecha) → **Settings → Developer → Access tokens → Generate new token**.
4. Agrega dos secretos en Colab:
   - `DATABRICKS_HOST`: la dirección de tu workspace, por ejemplo `https://dbc-1234abcd-5678.cloud.databricks.com`
   - `DATABRICKS_TOKEN`: el token que acabas de crear

Para hablar con Databricks desde afuera usamos su **API de archivos** (Files API). Para listar una carpeta se hace un `GET` a:

```
{DATABRICKS_HOST}/api/2.0/fs/directories{ruta}
```

con el token en el encabezado `Authorization: Bearer {token}`.

In [ ]:
DATABRICKS_HOST = userdata.get("DATABRICKS_HOST").strip().rstrip("/")
if not DATABRICKS_HOST.startswith("http"):
    DATABRICKS_HOST = "https://" + DATABRICKS_HOST

DB_HEADERS = {"Authorization": f"Bearer {userdata.get('DATABRICKS_TOKEN').strip()}"}
VOLUMEN = "/Volumes/workspace/dlbd/raw"

In [ ]:
# TODO: haz un GET a la Files API para listar el contenido de VOLUMEN.
# Usa DATABRICKS_HOST, VOLUMEN y DB_HEADERS. Guarda la respuesta en r.
r = None

In [ ]:
assert r is not None, "Completa la celda anterior."
assert r.status_code not in (401, 403), "Databricks rechazó el token. Revisa DATABRICKS_TOKEN."
assert r.status_code != 404, "No existe el volumen. ¿Ejecutaste el CREATE SCHEMA y el CREATE VOLUME?"
r.raise_for_status()

contenido = [c["name"] for c in r.json().get("contents", [])]
resultados["Databricks"] = True
print(f"✅ Databricks: {VOLUMEN} existe ({len(contenido)} elementos)")

## 4. Resumen y entrega

Ejecuta la celda de resumen. Si todo está en ✅, guarda este notebook **con sus salidas** en tu repositorio:
**Archivo → Guardar una copia en GitHub**, elige tu repositorio y la ruta `lab1/parte1_verificacion.ipynb`.

Ese guardado es la verificación de Colab: el notebook ejecutado debe aparecer en tu repositorio.

In [ ]:
for herramienta in ["GitHub", "Hugging Face", "Databricks"]:
    print(("✅" if resultados.get(herramienta) else "❌"), herramienta)
print("⬜ Colab: guarda este notebook en GitHub con sus salidas")